# 2. Regressione logistica e ipotesi di ricerca

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FedeGambe/Master_s_thesis_Data_science/blob/main/notebooks/02_regressione_logistica.ipynb)

Verifica delle ipotesi H1-H4 (capitoli 4-5 della tesi) con un modello logit. Rispetto alla versione originale: VIF e pseudo-R² calcolati con l'intercetta, intervalli di confidenza degli odds ratio, effetti marginali medi e variabili di percorrenza riscalate per avere OR leggibili.

In [ ]:
# Configurazione dell'ambiente (locale o Google Colab)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("Master_s_thesis_Data_science"):
        !git clone -q -b main https://github.com/FedeGambe/Master_s_thesis_Data_science.git
    %cd Master_s_thesis_Data_science
    !pip install -q -r requirements-colab.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import plotly.io as pio
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)
if os.environ.get("TESI_GRAFICI_STATICI"):
    pio.renderers.default = "png"  # usato solo per salvare output visibili su GitHub
    pio.defaults.default_width = 1000

## Ipotesi

| Codice | Ipotesi |
|---|---|
| **H1** | Un reddito più alto aumenta la probabilità di possedere un BEV |
| **H2.1** | Aver posseduto una PHEV favorisce il passaggio al BEV |
| **H2.2** | Come cambia la probabilità per chi aveva un'auto ICE, HEV o GNC? (esplorativa, senza direzione attesa) |
| **H2.3** | Chi ha già avuto un BEV tende a ricomprarlo |
| **H3** | Lunghe percorrenze riducono la probabilità di possedere un BEV |
| **H4** | Chi dà importanza alla riduzione delle emissioni sceglie più spesso un BEV |

In [ ]:
from tesi_bev import config as C, dati, statistica as S, grafici as G
df = dati.carica_dati()
X_completo, y = dati.prepara_logit(df, ridotto=False)

## 2.1 Multicollinearità (VIF)

Il VIF è calcolato con l'intercetta: nessuna variabile supera 3. Il modello stimato è quello della tesi, che esclude
*Numero persone in famiglia* e *Casa indipendente* (correlate con *Numero di auto* e *Casa di proprietà*), e viene
confrontato con il modello completo: la capacità esplicativa è praticamente la stessa.

In [ ]:
S.vif(X_completo)

## 2.2 Stima del modello

In [ ]:
X, y = dati.prepara_logit(df, ridotto=True)
risultato = S.logit(X, y)
pd.DataFrame({"completo": S.bonta_adattamento(S.logit(X_completo, y)), "ridotto": S.bonta_adattamento(risultato)})

In [ ]:
tabella = S.tabella_logit(risultato)
tabella

In [ ]:
G.forest_plot(tabella).show()

## 2.3 Effetti marginali medi

L'odds ratio non è una variazione di probabilità (nella tesi un OR di 1,22 era letto come "+22% di probabilità").
L'effetto marginale medio indica di quanti **punti percentuali** cambia in media la probabilità di possedere un BEV.

In [ ]:
S.effetti_marginali(risultato).sort_values('effetto_marginale')

## 2.4 Verifica delle ipotesi

In [ ]:
ipotesi = S.verifica_ipotesi(risultato)
ipotesi[["ipotesi", "variabile", "odds_ratio", "p_value", "direzione_attesa", "esito"]]

**Lettura.** H1, H2.3 e H4 sono confermate. H3 è confermata da numero di viaggi lunghi, VMT e distanza casa-lavoro
(quest'ultima al limite della significatività), ma non dal viaggio più lungo. **H2.1 non è supportata**: chi aveva
una PHEV ha una probabilità *minore* di possedere un BEV rispetto a chi aveva un'auto tradizionale (ICE).
**H2.2** (esplorativa): rispetto alla PHEV, aver avuto un'auto ICE aumenta la probabilità di possedere un BEV, mentre
HEV e GNC non differiscono in modo significativo.